# Real-Time Global Social Media Trend Analysis Using PySpark

**Goal**: Process a continuous, real-time data stream of text targeting Big Data scalability using built-in PySpark MLlib features. This notebook dynamically scrapes live global feeds to process real-time events across Sports, World News, Tech, and Entertainment!

### Step 1: Create the Web-Scraping Streaming Service
Run this cell to create a script that connects directly to Global RSS sources. It relies on extremely stable endpoints so it completely bypasses Colab's network blocking.

In [ ]:
%%writefile data_stream_generator.py
import os
import json
import time
import uuid
import shutil
import urllib.request
import xml.etree.ElementTree as ET
import random

OUTPUT_DIR = "stream_input"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Starting Multi-Sector Global Live Feed... Writing to '{OUTPUT_DIR}'")
headers = {'User-Agent': 'Mozilla/5.0'}

# Pulling from 4 completely separate global sectors to ensure we aren't just looking at Tech!
FEEDS = {
    "News": "http://feeds.bbci.co.uk/news/world/rss.xml",
    "Entertainment": "https://www.eonline.com/syndication/feeds/rssfeeds/topstories.xml",
    "Sports": "https://www.espn.com/espn/rss/news",
    "Technology": "http://feeds.bbci.co.uk/news/technology/rss.xml"
}

def fetch_rss_feed(url):
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=5) as response:
            tree = ET.fromstring(response.read())
            items = tree.findall('.//item')
            fetched = []
            for item in items[:15]: 
                title = item.find('title')
                desc = item.find('description')
                title_text = title.text if title is not None else ""
                desc_text = desc.text if desc is not None else ""
                
                text = f"{title_text} - {desc_text}".replace("<p>", "").replace("</p>", "").replace("&nbsp;", " ").strip()[:300]
                
                # DYNAMIC PRECISE HASHTAGS: We extract capitalized nouns dynamically as our hashtags! 
                words = [w.strip(".,;:!'\"?") for w in title_text.split()]
                tags = ["#" + w for w in words[1:] if w and w[0].isupper() and len(w) > 3]
                tag_str = " ".join(list(set(tags))[:3]) # Top 3 unique capital words
                
                fetched.append({"id": str(uuid.uuid4()), "text": f"{text} {tag_str}"})
            return fetched
    except Exception as e:
        return []

try:
    file_counter = 1
    while True:
        tweets = []
        for tag, url in FEEDS.items():
            tweets.extend(fetch_rss_feed(url))
            
        random.shuffle(tweets)
        
        if tweets:
            # Mix it up per batch
            batch = random.sample(tweets, k=random.randint(10, 25))
            
            # ATOMIC WRITES to ensure PySpark picks it up cleanly
            tmp_path = os.path.join(OUTPUT_DIR, f"batch_{file_counter}.tmp")
            final_path = os.path.join(OUTPUT_DIR, f"batch_{file_counter}.json")
            
            with open(tmp_path, "w", encoding="utf-8") as f:
                for t in batch:
                    f.write(json.dumps(t) + "\n")
            
            os.rename(tmp_path, final_path)
            file_counter += 1
            
        time.sleep(5) 
except KeyboardInterrupt:
    pass


### Step 2: Start the Generator in the Background
Run this to start pulling live data silently without holding up the notebook.

In [ ]:
!nohup python data_stream_generator.py > generator_logs.txt 2>&1 &

### Step 3: PySpark Imports & Initialization

In [ ]:
!pip install pyspark textblob matplotlib pandas ipython

import time
import IPython
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.sql.functions import col, explode, udf, desc
from pyspark.ml.feature import Tokenizer, StopWordsRemover
from textblob import TextBlob
import re

spark = SparkSession.builder \
    .appName("RealTimePrecisionAnalysis") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("PySpark Structured Streaming Ready!")

### Step 4: Establish the Data Stream Source

In [ ]:
schema = StructType([
    StructField("id", StringType(), True),
    StructField("text", StringType(), True)
])

raw_stream_df = spark.readStream \
    .schema(schema) \
    .json("stream_input")

print("Is DataFrame streaming?", raw_stream_df.isStreaming)

### Step 5: Big Data Preprocessing (PySpark MLlib)

In [ ]:
def clean_punctuation(text):
    if not text: return ""
    words = re.findall(r'#\w+|\w+', text.lower())
    return " ".join(words)

clean_punct_udf = udf(clean_punctuation, StringType())
cleaned_df = raw_stream_df.withColumn("clean_string", clean_punct_udf(col("text")))

tokenizer = Tokenizer(inputCol="clean_string", outputCol="raw_words")
tokenized_df = tokenizer.transform(cleaned_df)

remover = StopWordsRemover(inputCol="raw_words", outputCol="filtered_words")
processed_df = remover.transform(tokenized_df)

### Step 6: Add Sentiment & Aggregate Streams

In [ ]:
def get_sentiment(text):
    if not text: return 0.0
    return TextBlob(text).sentiment.polarity

def classify_sentiment(polarity):
    if polarity > 0.1: return "Positive"
    elif polarity < -0.1: return "Negative"
    else: return "Neutral"

sentiment_val_udf = udf(get_sentiment, FloatType())
sentiment_class_udf = udf(classify_sentiment, StringType())

enriched_df = processed_df \
    .withColumn("polarity", sentiment_val_udf(col("text"))) \
    .withColumn("sentiment", sentiment_class_udf(col("polarity")))

exploded_df = enriched_df.withColumn("word", explode(col("filtered_words")))
hashtag_stream = exploded_df.filter(col("word").startswith("#"))

hashtag_counts = hashtag_stream.groupBy("word").count()
sentiment_counts = enriched_df.groupBy("sentiment").count()

### Step 7: Start the Analytics Engine

In [ ]:
query_hashtags = hashtag_counts \
    .writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hashtags_table") \
    .start()

query_sentiments = sentiment_counts \
    .writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("sentiments_table") \
    .start()

query_raw = raw_stream_df \
    .writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("raw_table") \
    .start()

print("Streaming Queries are ACTIVE! Processing incoming data LIVE...")

### Step 8: Gorgeous UI Live Dashboard (Dark Mode) & Real-time Text Feeds

In [ ]:
try:
    time.sleep(4)
    while True:
        top_hashtags_pd = spark.sql("SELECT * FROM hashtags_table ORDER BY count DESC LIMIT 8").toPandas()
        sentiment_pd = spark.sql("SELECT * FROM sentiments_table").toPandas()

        IPython.display.clear_output(wait=True)
        
        # ------------------ STYLING UPGRADES ------------------
        plt.style.use('dark_background')
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        fig.patch.set_facecolor('#1e1e1e')
        ax1.set_facecolor('#1e1e1e')
        ax2.set_facecolor('#1e1e1e')
        
        bar_colors = ['#FF3366', '#FF6633', '#FFCC33', '#33CC99', '#3399FF', '#9933FF', '#FF33CC', '#00FF99']
        pie_colors = ['#00E676', '#FF1744', '#757575'] # Green, Red, Grey
        
        if not top_hashtags_pd.empty:
            bars = ax1.bar(top_hashtags_pd['word'], top_hashtags_pd['count'], color=bar_colors[:len(top_hashtags_pd)], edgecolor='none')
            
            # Add data labels on top of bars
            for bar in bars:
                yval = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.5, int(yval), ha='center', va='bottom', color='#FFFFFF', fontweight='bold')

            ax1.set_title('🔥 LIVE TRENDING: GLOBAL EVENTS', fontsize=18, fontweight='bold', color='#FFFFFF')
            ax1.set_ylabel('Total Mentions', fontsize=12, color='#CCCCCC')
            ax1.tick_params(axis='x', rotation=45, colors='#CCCCCC', labelsize=11)
            ax1.tick_params(axis='y', colors='#CCCCCC')
            ax1.grid(axis='y', linestyle='--', alpha=0.3)
            ax1.spines['top'].set_visible(False)
            ax1.spines['right'].set_visible(False)
        else:
            ax1.set_title('🔥 LIVE TRENDING: GLOBAL EVENTS (Syncing...)', fontsize=18, color='#FFFFFF')

        if not sentiment_pd.empty:
            wedges, texts, autotexts = ax2.pie(sentiment_pd['count'], labels=sentiment_pd['sentiment'], autopct='%1.1f%%', 
                    colors=pie_colors, startangle=140, explode=[0.05]*len(sentiment_pd), shadow=True)
            plt.setp(autotexts, size=11, weight="bold", color="white")
            plt.setp(texts, size=12, color="white", weight="bold")
            ax2.set_title('🧠 CUMULATIVE GLOBAL SENTIMENT', fontsize=18, fontweight='bold', color='#FFFFFF')
        else:
            ax2.set_title('🧠 CUMULATIVE GLOBAL SENTIMENT (Syncing...)', fontsize=18, color='#FFFFFF')
            
        plt.tight_layout()
        plt.show()
        
        # ------------------ RAW DATA PRINTER ------------------
        try:
            raw_pd = spark.sql("SELECT text FROM raw_table ORDER BY id DESC LIMIT 5").toPandas()
            if not raw_pd.empty:
                print("\n\033[1m" + "="*90)
                print("\U0001f30e LIVE INCOMING GLOBAL DATA STREAM (News, Sports, Entertainment):")
                print("="*90 + "\033[0m")
                for text in raw_pd['text']:
                    print(f"🔹 {text[:250]}...\n")
        except:
            pass

        time.sleep(3)
except KeyboardInterrupt:
    print("Dashboard updates stopped by user.")

In [ ]:
query_hashtags.stop()
query_sentiments.stop()
query_raw.stop()
spark.stop()
print("Spark streaming engines halted.")